# Urdu OCR — SFT then RL

Trains `Qwen/Qwen3.5-4B` to transcribe Urdu newspaper clippings, in two stages,
and measures whether the RL stage adds anything over SFT alone.

The corpus is the **Urdu Newspaper Benchmark** (829 paragraph images, 9,982
lines) from [*From Press to Pixels*](https://arxiv.org/abs/2505.13943). Published
reference points on this exact test set, so results here are comparable rather
than freestanding:

| system | CER |
|---|---:|
| TrOCR — best specialist baseline | 0.159 |
| **Qwen3.5-4B zero-shot, high-res** | **0.168** |
| Gemini-2.5-Pro zero-shot | 0.032 |

The 4B base model already sits at the specialist baseline. **Beating 0.159 is the
bar for "decent Urdu OCR from a small open model."**

## Two corrections from RL_review_v1.md, both applied here

**Resolution.** Every prior run used the low-res capture, because the original
task framing was *correcting* a low-res first pass. That stopped being the task
and nothing caught it. Over all 829 documents, same model, same prompt:

| input | CER | sub | ins |
|---|---:|---:|---:|
| low-res, 4× upscale | 0.332 | 0.155 | 0.240 |
| **high-res, 2× upscale** | **0.168** | **0.068** | **0.117** |

Better on 96.1% of documents, with substitutions halved — the model reads better,
not merely shorter. This notebook uses `rpant/ocr-urdu-splits-hr`, whose document
assignment is identical to the low-res splits.

**The vision tower trains too** (`lora_vision=True`). For a script the base model
barely saw in pretraining, freezing the encoder is the wrong default; the Manchu
low-resource work tuned both towers and reached CER 0.0219 on real documents.
Language-only is now the ablation, not the baseline.

## Why this design

Prior RL-only runs on Prime hosted **failed twice**, with different reward shapes:

| run | what happened |
|---|---|
| 100-step, cliff length penalty | CER 0.337 → 0.585, length ratio 0.93 → 2.17, then 51% empty rollouts |
| 50-step, smooth penalty + higher cap | CER 0.344 → 0.972, substitution rate 0.166 → 0.227 at matched length |

The diagnosis was not the reward function. `pass@16` on the base policy gave
best-of-16 CER **0.247** against a greedy baseline of **0.262** — a tail worth
2.1% of available headroom. GRPO can only amplify behaviour the policy already
samples, so with a tail that thin the advantage signal is dominated by length
noise and updates random-walk downhill.

SFT has no such precondition: dense per-token targets install behaviour rather
than amplifying it. A frontier VLM reads these same images at **0.022** CER, so
the information is in the pixels and the gap is model capability.

## Three arms, one held-out test set

| arm | training | question |
|---|---|---|
| **A** | SFT on `sft_train` (479) | does SFT move CER at all? |
| **B** | A → **GRPO** on `rl_train` (250) | does RL add value? |
| **C** | A → **more SFT** on `rl_train` (250) | …or would the same data as SFT do just as well? |

**B vs C is the actual experiment.** Comparing A to B alone confounds "RL helped"
with "250 more documents helped". C holds the data budget fixed.

All three are scored on the same 100 test documents, **paired per document**,
which removes between-document variance (CER std across documents is ~0.18, far
larger than the effects we are chasing).

Splits are disjoint by construction — see `data_prep/make_splits.py`, which
asserts non-overlap before pushing.

If you have decided against RL entirely, set `train_splits = ("sft_train",
"rl_train")` in §2 for **729** training documents instead of 479, and stop after
§8. Arms B and C become invalid in that configuration and the notebook refuses to
run them.

## Before the RL stage, there is a gate

Cell 9 re-runs `pass@16` on the SFT'd model and reports **greedy − best-of-16**.
That number was measured to track the thing that actually matters: on the 35B
hosted run it was 0.028 before training, and the advantage standard deviation at
step 1 of that run was 0.032 — agreement within 15%. The advantage spread *is* the
learning signal, so this probe estimates it for a few minutes of generation
instead of a 50-step run.

Below 0.05, RL will fail for the same reason it failed three times already, and
the notebook says so rather than burning an hour finding out.

---
**Runtime:** A100 40GB (Colab Pro). Set Runtime → Change runtime type → A100.

**Hugging Face access is required.** The splits are private datasets, so an
anonymous `load_dataset` returns HTTP 401, which `datasets` reports as the
misleading *"Dataset ... doesn't exist on the Hub or cannot be accessed"*. §1
resolves a token before anything else runs. Easiest path: Colab's 🔑 sidebar →
add a secret named `HF_TOKEN` (a read token from
[hf.co/settings/tokens](https://huggingface.co/settings/tokens)) → enable
notebook access. It then persists across sessions.

## 1 · Environment

In [ ]:
# Colab: A100 40GB. Check what we actually got -- Colab silently downgrades.
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Install ONLY what Colab lacks. pillow/matplotlib/pandas/scipy/numpy ship with
# the runtime already, and `-U` on them is actively harmful: Colab's base image
# does not replace every file atomically, so upgrading pillow in place leaves a
# new ImageDraw.py against a stale _typing.py and the next import dies with
#   ImportError: cannot import name '_Ink' from 'PIL._typing'
# Nothing here needs a newer pillow than the runtime provides.
# Colab preinstalls torchao 0.10.0. Recent PEFT probes its TorchAO LoRA
# dispatcher during adapter injection and *raises* on a too-old version instead
# of falling back to ordinary LoRA:
#   ImportError: Found an incompatible version of torchao. Found version 0.10.0,
#   but only versions above 0.16.0 are supported
# It surfaces deep in section 7, at attach_lora(). Nothing here uses torchao, so
# remove it rather than upgrading -- torchao is pinned tightly to the torch build
# and upgrading trades this mismatch for a harder one.
%pip -q uninstall -y torchao

%pip -q install -U "transformers>=4.57" "trl>=0.24" "peft>=0.17" "datasets>=3.0" \
    "accelerate>=1.0" "bitsandbytes>=0.44" jiwer

import importlib.util
if importlib.util.find_spec("torchao") is not None:
    print("!! torchao is still importable -- a dependency reinstalled it.")
    print("   Runtime -> Restart session before continuing, or attach_lora() will raise.")
else:
    print("torchao absent (intended: PEFT then takes the plain LoRA path)")

import PIL, importlib
print("pillow", PIL.__version__, "(runtime default, deliberately not upgraded)")
try:
    from PIL import ImageDraw, ImageFilter          # the imports that break when split
    importlib.import_module("PIL._typing")
    print("pillow imports clean")
except ImportError as e:
    raise SystemExit(
        f"pillow is in a half-upgraded state: {e}\n"
        "Runtime -> Restart session, then run this cell again. Re-installing "
        "will not help; the stale module is already loaded in this process.")
print("\nIf transformers was imported earlier this session, restart the runtime.")

### 1a · Hugging Face access

The splits are private. Without a token `load_dataset` gets HTTP 401 and reports
it as *"doesn't exist on the Hub or cannot be accessed"* — which reads like a
typo in the dataset name and is not. Resolved here so it fails in 5 seconds
rather than after the model has loaded.

In [ ]:
import os

def _cached_token():
    """Token from the huggingface-cli cache, across library versions.

    `HfFolder.get_token()` was the old spelling and is gone in current
    huggingface_hub; `get_token()` is the replacement. Try the new name first
    and fall back, so this works on whatever Colab happens to install.
    """
    try:
        from huggingface_hub import get_token
        return get_token()
    except ImportError:
        pass
    try:
        from huggingface_hub import HfFolder
        return HfFolder.get_token()
    except Exception:
        return None

def resolve_hf_token():
    """Colab secret -> environment -> cli cache -> interactive login. First hit wins."""
    try:                                    # Colab Secrets (persists across sessions)
        from google.colab import userdata
        t = userdata.get("HF_TOKEN")
        if t: print("token source: Colab secret HF_TOKEN"); return t.strip()
    except Exception:
        pass
    for var in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN", "HUGGINGFACEHUB_API_TOKEN"):
        if os.environ.get(var):
            print(f"token source: ${var}"); return os.environ[var].strip()
    t = _cached_token()
    if t: print("token source: huggingface-cli cache"); return t.strip()
    print("no token found -- opening interactive login")
    from huggingface_hub import notebook_login
    notebook_login()
    t = _cached_token()
    if not t:
        raise RuntimeError("login did not produce a token; add a Colab secret "
                           "named HF_TOKEN instead (sidebar key icon)")
    return t.strip()

HF_TOKEN = resolve_hf_token()
os.environ["HF_TOKEN"] = HF_TOKEN or ""     # datasets/transformers also read this

# Verify against the dataset we actually need, not just "is the token valid".
# A valid token belonging to the wrong account fails exactly here, with a
# message that says so.
from huggingface_hub import HfApi
_api = HfApi(token=HF_TOKEN)
print("logged in as:", _api.whoami().get("name", "?"))

In [ ]:
DATASET_ID = "rpant/ocr-urdu-splits-hr"     # must match CFG["dataset_id"] in section 2
try:
    info = _api.dataset_info(DATASET_ID)
    print(f"OK  {info.id}  private={info.private}  files={len(info.siblings)}")
except Exception as e:
    raise SystemExit(
        f"cannot reach {DATASET_ID}: {type(e).__name__}: {e}\n\n"
        "Checklist:\n"
        f"  1. Does it exist?  https://huggingface.co/datasets/{DATASET_ID}\n"
        "  2. Is your token from the account that owns it (printed above)?\n"
        "  3. Does the token have read scope?\n"
        "  4. In Colab: sidebar key icon -> HF_TOKEN -> 'Notebook access' toggled ON.\n"
        "  5. Not built yet? Run, from the repo:\n"
        "     uv run python data_prep/package_dataset.py --image-source high "
        "--out packaged_dataset_hr\n"
        "     uv run python data_prep/make_splits.py --dataset packaged_dataset_hr "
        f"--out splits_dataset_hr --push-to-hub {DATASET_ID} --private")

In [ ]:
import os, json, math, time, random, gc, re, unicodedata, io, base64
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np, pandas as pd, torch
from PIL import Image
import matplotlib.pyplot as plt

def seed_all(s=1234):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")

## 2 · Configuration

Every knob lives here.

`USE_AUGMENTATION` is **off by default**, and when enabled it applies to **arm A
only** (`sft_train`). Arms B and C both operate on the raw 250 `rl_train`
documents, because B-vs-C is a comparison of *method* at a fixed data budget —
augmenting one side would make it a comparison of dataset size instead.

Augmentation is also deliberately kept out of the RL stage on its own merits: it
adds input variance, which adds reward variance, which degrades the GRPO
advantage estimate. That signal is already the weakest link here.

In [ ]:
CFG = dict(
    # --- data ---
    # HIGH-RES splits. The low-res twin (`rpant/ocr-urdu-splits`) is what every
    # run in RL_report_v1.md used, and it was the wrong input: the corpus ships a
    # 4x-larger capture per document, and packaging only embedded the small one.
    # Paired over all 829 docs, same Qwen3.5-4B, same prompt:
    #     low-res  4x upscale   CER 0.332   sub 0.155   ins 0.240
    #     high-res 2x upscale   CER 0.168   sub 0.068   ins 0.117
    # High-res wins on 96.1% of documents. Substitutions halve, so the model
    # reads better rather than merely writing less. Document assignment is
    # byte-identical to the low-res splits (same seed), so anything already
    # measured stays comparable.
    dataset_id     = "rpant/ocr-urdu-splits-hr",  # sft_train 479 / rl_train 250 / test 100
    image_upscale  = 2,      # the measured high-res operating point
    image_format   = "jpeg", # 468KB PNG -> 133KB JPEG per example, same CER
    image_quality  = 90,

    # Which splits feed arm A. Arms B and C need `rl_train` held back, so the
    # default keeps it out. If you have decided against RL entirely, set this to
    # ("sft_train","rl_train") for 729 training documents instead of 479 -- a 52%
    # increase -- and skip sections 9-12. One-way door for this run only, since
    # the split assignment lives in the dataset and is reproducible.
    train_splits   = ("sft_train"),

    # --- augmentation (OFF by default; ARM A / sft_train ONLY) ---
    USE_AUGMENTATION   = False,
    aug_multiplier     = 2,          # extra augmented copies per source document
    aug_upscale_choices= (3, 4, 5),
    aug_quality_range  = (75, 95),
    aug_rotate_deg     = 1.0,        # +/- degrees; newspapers are near-axis-aligned
    aug_blur_prob      = 0.3,

    # --- model ---
    base_model     = "Qwen/Qwen3.5-4B",
    attn_impl      = "sdpa",
    dtype          = "bfloat16",

    # --- LoRA ---
    lora_r         = 32,
    lora_alpha     = 64,
    lora_dropout   = 0.05,
    # Language-side projection names. The vision tower is added separately at
    # runtime (see `resolve_lora_targets`) because its module names differ across
    # VLM generations and cannot be guessed offline.
    lora_targets   = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    # Train the vision encoder too. RL_report_v1.md filed this as an open
    # question; the low-resource-script literature answers it. The Manchu work
    # (Qwen2.5-VL 3B/7B, LLaMA-3.2-11B) applied PEFT to "both vision and language
    # components" at rank 64 and reached CER 0.0219 on real historical documents.
    # For a script the base model barely saw in pretraining, a frozen encoder is
    # the wrong default -- so language-only is the ablation here, not the baseline.
    lora_vision    = True,

    # --- SFT (arm A) ---
    sft_epochs         = 3,
    sft_lr             = 1e-4,
    sft_batch          = 1,
    sft_grad_accum     = 8,
    sft_warmup_ratio   = 0.03,
    # A cap, not an allocation: the collator pads to the longest item in the
    # batch (padding=True), and batch size is 1, so tensors are sized by actual
    # content and a generous cap costs no memory. Truncation that lands inside
    # the image-token block *raises* rather than degrading, so the cap must clear
    # the longest real sequence with margin.
    # Measured over all 829 documents: Urdu tokenizes at 0.652 tokens/char, so
    # targets run median 287, p90 726, max 1,563 tokens; plus ~50 for the
    # instruction and chat template. With max_image_tokens below, worst case is
    # ~4,096 + 1,563 + 50 = 5,709.
    # Section 7's preflight measures this over every example instead of trusting
    # the arithmetic -- an earlier version of this comment held image tokens
    # constant at 2,496 and was 5x low on the tail.
    # Per-page image token budget, enforced as a pixel cap (see
    # cap_image_pixels). Without it, token count follows the source scan's area
    # and reaches 12,060 on the largest page at upscale 2.
    max_image_tokens   = 4096,
    sft_max_seq_len    = 8192,
    sft_eval_every     = 25,     # steps; evaluates on EVAL_SUBSET of test
    sft_eval_subset    = 32,     # keep small -- generation is the slow part

    # --- arm C (more SFT on rl_train) ---
    c_epochs           = 3,
    c_lr               = 5e-5,   # lower: continuing from an already-tuned adapter

    # --- GRPO (arm B) ---
    grpo_steps         = 60,
    grpo_num_generations = 8,    # rollouts per prompt (the GRPO group)
    grpo_lr            = 5e-6,   # deliberately low; prior runs drifted at default
    grpo_batch         = 8,
    grpo_grad_accum    = 4,
    grpo_temperature   = 0.7,
    grpo_max_new       = 1024,
    grpo_beta          = 0.04,   # KL to reference. Prior hosted runs used ~1e-3
                                 # and drifted; this anchors the policy far harder.

    # --- generation / eval ---
    gen_max_new        = 1024,
    passk_k            = 16,
    passk_docs         = 20,

    # --- output ---
    # Drive holds what must survive the session: metrics, figures, and the final
    # adapters (~110 MB each at bf16).
    run_dir            = "/content/drive/MyDrive/ocr_sft_rl",
    # Trainer checkpoints go to LOCAL disk instead. They carry optimizer state --
    # roughly 3x the adapter size each -- and exist only for crash recovery
    # inside a session, so writing them to Drive burns ~700 MB of quota for
    # files nothing ever reads. Running out mid-run means nothing lands at all.
    ckpt_dir           = "/content/ocr_ckpt",
    seed               = 1234
)

INSTRUCTION = ("Transcribe all text in this document image. "
               "Preserve line breaks. Output only the transcription.")

# Mount Drive so a Colab disconnect does not lose the run.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("not on Colab or mount skipped:", e)
    CFG["run_dir"] = "./ocr_sft_rl"

RUN = Path(CFG["run_dir"]); (RUN/"metrics").mkdir(parents=True, exist_ok=True)
(RUN/"adapters").mkdir(exist_ok=True); (RUN/"figures").mkdir(exist_ok=True)
(RUN/"config.json").write_text(json.dumps(CFG, indent=2))
CKPT = Path(CFG["ckpt_dir"]); CKPT.mkdir(parents=True, exist_ok=True)
print("run dir (persistent):", RUN)
print("ckpt dir (ephemeral):", CKPT)

# Free space matters more than it looks: Drive filling up mid-run means the
# final adapter and the metrics both fail to write, so the whole session is
# lost rather than degraded. Check before spending an hour on an A100.
import shutil
for label, p in (("run_dir", RUN), ("ckpt_dir", CKPT)):
    free_gb = shutil.disk_usage(p).free / 2**30
    print(f"  {label}: {free_gb:.1f} GB free")
    if label == "run_dir" and free_gb < 1.0:
        print("   !! under 1 GB. Each adapter is ~110 MB and metrics add up; "
              "free space on Drive before training.")

## 3 · Metrics

CER is decomposed into **substitutions / deletions / insertions**, all normalised
by reference length. This decomposition is what made the RL failures legible:
over-generation appears *only* in insertions, misreading glyphs *only* in
substitutions. Watching CER alone cannot tell those apart, and they call for
opposite fixes.

Normalization matches the Prime environments exactly (`ocr_direct.py`) so numbers
remain comparable across platforms.

In [ ]:
import jiwer

_NORM = str.maketrans({
    "\u064A": "\u06CC",  # Arabic yeh   -> Farsi yeh
    "\u0649": "\u06CC",  # alef maksura -> Farsi yeh
    "\u0643": "\u06A9",  # Arabic kaf   -> Keheh
    "\u0640": "",          # tatweel
    "\u200C": " ",         # ZWNJ
    "\u200D": "", "\u200E": "", "\u200F": "",
})
CER_CAP = 1.5

def normalize(s):
    s = unicodedata.normalize("NFC", s or "")
    s = s.translate(_NORM)
    return re.sub(r"\s+", " ", s).strip()

_FENCE = re.compile(r"^\s*```[a-zA-Z]*\s*\n(.*?)\n?\s*```\s*$", re.DOTALL)
_PREAMBLE = re.compile(r"^\s*(here (is|are)[^:\n]*:|the transcription[^:\n]*:|transcription:)\s*", re.I)
# Reasoning blocks, closed or not. `render_chat` disables thinking at the
# template level, so this should never fire -- but if a template ignores the
# flag, reasoning text would otherwise be scored as inserted characters and
# show up as a reading failure instead of a configuration one. The unterminated
# alternative matters because a block truncated by max_new_tokens has no
# closing tag.
_THINK = re.compile(r"<think>.*?</think>|<think>.*\Z", re.DOTALL | re.I)

def clean_output(text):
    if not text: return ""
    text = _THINK.sub("", text)
    m = _FENCE.match(text)
    if m: text = m.group(1)
    return _PREAMBLE.sub("", text).strip()

def cer_breakdown(ref, hyp):
    """Capped CER + sub/del/ins rates. sub+del+ins == uncapped CER."""
    ref, hyp = normalize(ref), normalize(hyp)
    if not ref:  return dict(cer=0.0 if not hyp else CER_CAP, sub=0., dele=0., ins=0., ratio=0.)
    if not hyp:  return dict(cer=CER_CAP, sub=0., dele=1., ins=0., ratio=0.)
    o = jiwer.process_characters(ref, hyp); n = max(1, len(ref))
    return dict(cer=min(float(o.cer), CER_CAP), sub=o.substitutions/n,
                dele=o.deletions/n, ins=o.insertions/n, ratio=len(hyp)/n)

def summarize(records):
    # Bracket access throughout: `d.sub` resolves to DataFrame.sub (subtraction),
    # not the column, and silently returns a bound method. Same trap waits on
    # `mean`, `std`, `min`, `max`.
    d = pd.DataFrame(records)
    return dict(n=len(d), cer=d["cer"].mean(), cer_median=d["cer"].median(),
                sub=d["sub"].mean(), dele=d["dele"].mean(), ins=d["ins"].mean(),
                ratio=d["ratio"].mean(), empty=float((d["ratio"] == 0).mean()))

def jdump(obj, path):
    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False)); print("wrote", path)

def jsonl_append(obj, path):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

# sanity-check the metric on synthetic cases before trusting it on real output
_gt = "\u06c1\u0645\u0627\u0631\u06d2 \u062f\u0641\u0627\u062a\u0631 \u0627\u0648\u0631"
for name, hyp in [("perfect", _gt), ("2x", _gt+" "+_gt), ("empty", "")]:
    print(f"  {name:<8}", {k: round(v,3) for k,v in cer_breakdown(_gt, hyp).items()})

## 4 · Data

Splits load straight from the Hub. `sft_train`, `rl_train` and `test` are
guaranteed disjoint — re-asserted here rather than assumed, because contamination
between these three is the one error that would invalidate every number below.

In [ ]:
from datasets import load_dataset

assert CFG["dataset_id"] == DATASET_ID, (
    f"section 1a verified access to {DATASET_ID!r} but CFG points at "
    f"{CFG['dataset_id']!r} -- update both or the check was meaningless")

SPLITS = {s: load_dataset(CFG["dataset_id"], split=s, token=HF_TOKEN)
          for s in ["sft_train","rl_train","test"]}
ids = {k: set(v["doc_id"]) for k,v in SPLITS.items()}
for a in ids:
    for b in ids:
        assert not (a < b and ids[a] & ids[b]), f"CONTAMINATION: {a} n {b}"
print({k: len(v) for k,v in SPLITS.items()}, "| disjoint OK")

# Confirm we are on the high-res capture. Pixel dimensions cannot settle this --
# the two sources overlap in size and upscaling erases the difference -- so read
# the column the packaging step recorded.
src = set(SPLITS["sft_train"]["image_source"])
assert src == {"high"}, f"expected high-res images, got {src}; check CFG['dataset_id']"
w, h = SPLITS["sft_train"][0]["image"].size
print(f"image_source={src.pop()}  native size {w}x{h}  -> {w*CFG['image_upscale']}x{h*CFG['image_upscale']} after upscale")

gt_lens = [len(x) for x in SPLITS["sft_train"]["ground_truth"]]
print(f"ground-truth chars: min {min(gt_lens)} median {int(np.median(gt_lens))} "
      f"p90 {int(np.percentile(gt_lens,90))} max {max(gt_lens)}")

In [ ]:
def encode_image(img: Image.Image, upscale=None, quality=None, fmt=None, augment=False):
    """Upscale + re-encode. Returns a PIL image (the processor wants PIL, not a data URL)."""
    upscale = upscale or CFG["image_upscale"]
    quality = quality or CFG["image_quality"]
    fmt     = fmt or CFG["image_format"]
    im = img.convert("RGB")
    if augment:
        upscale = random.choice(CFG["aug_upscale_choices"])
        quality = random.randint(*CFG["aug_quality_range"])
        if CFG["aug_rotate_deg"]:
            im = im.rotate(random.uniform(-CFG["aug_rotate_deg"], CFG["aug_rotate_deg"]),
                           resample=Image.BILINEAR, fillcolor=(255,255,255))
        if random.random() < CFG["aug_blur_prob"]:
            from PIL import ImageFilter
            im = im.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.2, 0.6)))
    if upscale > 1:
        im = im.resize((im.width*upscale, im.height*upscale), Image.LANCZOS)
    # round-trip through the target codec so training sees the same artifacts as eval
    buf = io.BytesIO()
    im.save(buf, format="JPEG" if fmt in ("jpeg","jpg") else fmt.upper(), quality=quality)
    buf.seek(0)
    return Image.open(buf).convert("RGB")

def build_examples(split_name, augment=False, multiplier=1):
    """-> list of {image, target, doc_id}. Augmented copies are appended, never
    replacements, so the original clean view is always present.

    `split_name` accepts one split name or an iterable of them; multiple splits
    are concatenated in the order given.
    """
    names = [split_name] if isinstance(split_name, str) else list(split_name)
    rows = [ex for n in names for ex in SPLITS[n]]
    out = []
    for ex in rows:
        out.append(dict(image=encode_image(ex["image"]), target=ex["ground_truth"], doc_id=ex["doc_id"]))
    if augment:
        for _ in range(max(0, multiplier-1)):
            for ex in rows:
                out.append(dict(image=encode_image(ex["image"], augment=True),
                                target=ex["ground_truth"], doc_id=ex["doc_id"]+"_aug"))
    return out

t0=time.time()
SFT_EXAMPLES = build_examples(CFG["train_splits"], augment=CFG["USE_AUGMENTATION"],
                              multiplier=CFG["aug_multiplier"])
print(f"SFT examples: {len(SFT_EXAMPLES)} from splits {tuple(CFG['train_splits'])} "
      f"(augmentation={CFG['USE_AUGMENTATION']}) in {time.time()-t0:.0f}s")
print("image size after upscale:", SFT_EXAMPLES[0]["image"].size)

# Arms B and C compare method at a fixed data budget on `rl_train`. If arm A has
# already consumed it, that comparison no longer exists and sections 9-12 must be
# skipped rather than run on contaminated data.
RL_AVAILABLE = "rl_train" not in CFG["train_splits"]
if not RL_AVAILABLE:
    print("\n!! rl_train is in train_splits -> arms B and C are INVALID for this run.")
    print("   Stop after section 8 and read arm A against the test split.")

## 5 · Model + LoRA

⚠️ **Verify this cell before the long run.** Class names and processor behaviour
for a given VLM generation change between `transformers` releases. If the
auto-class fails, read the traceback — the fix is usually the model-specific
class named in the error.

`AutoModelForImageTextToText` replaced `AutoModelForVision2Seq`, which was
**removed in transformers v5** (Jan 2026). The new name exists back to 4.46, so
it is the safer spelling on either side of that boundary.

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel

DTYPE = getattr(torch, CFG["dtype"])

def cap_image_pixels(proc, budget_tokens=None):
    """Bound image tokens per page by capping total pixels.

    Qwen3.5's processor ships no pixel cap, and `image_upscale` is a blind
    multiplier, so token count tracks the source scan's area -- which varies 16x
    across this corpus (484x352 to 1468x2148). Measured at upscale 2:

        image tokens   median 1,550   p90 4,320   max 12,060

    A 12k-token page is ~30x the attention compute of the median one and
    overruns any sane sequence budget. patch_size 16 with merge_size 2 means one
    token per 32x32 pixel block, so a token budget converts directly to a pixel
    cap. Only the largest ~7% of pages are touched, and those have the most
    redundant detail; the median page is unaffected.

    Applied to the single shared processor, so training and evaluation see the
    same preprocessing -- a cap on one side only would confound the very
    before/after comparison this notebook exists to make.
    """
    budget = budget_tokens or CFG["max_image_tokens"]
    px = budget * 32 * 32
    try:
        proc.image_processor.size = {"longest_edge": px, "shortest_edge": 65536}
    except Exception as e:
        print(f"  !! could not set pixel cap ({e}); sequences may overrun")
    return proc

def load_base(adapter_path=None, for_training=True):
    proc = AutoProcessor.from_pretrained(CFG["base_model"], trust_remote_code=True)
    cap_image_pixels(proc)
    model = AutoModelForImageTextToText.from_pretrained(
        CFG["base_model"], torch_dtype=DTYPE, device_map="auto",
        attn_implementation=CFG["attn_impl"], trust_remote_code=True)
    if adapter_path:
        model = PeftModel.from_pretrained(model, adapter_path, is_trainable=for_training)
        print("loaded adapter:", adapter_path)
    return model, proc

VISION_PREFIXES = ("visual", "vision_tower", "vision_model", "image_encoder")
# PEFT's own injected submodules. Once LoRA is attached, named_modules() reports
# `...qkv.base_layer`, `...qkv.lora_A.default` etc, and those are real nn.Linear
# instances -- so enumerating a wrapped model yields LoRA internals as targets
# and PEFT then fails to resolve them. Skipped defensively; attach_lora also
# refuses to run twice.
PEFT_INTERNALS = ("base_layer", "lora_A", "lora_B", "lora_embedding_A",
                  "lora_embedding_B", "lora_magnitude_vector", "modules_to_save",
                  "original_module")

def resolve_lora_targets(model, include_vision=None):
    """Enumerate real nn.Linear module paths instead of guessing leaf names.

    Two reasons this is done at runtime rather than hardcoded in CFG:

    1. Vision-tower module names differ across VLM generations (`visual.blocks.
       *.attn.qkv` on Qwen2.5-VL, other layouts elsewhere). Guessing offline
       produces a config that silently matches nothing.
    2. PEFT resolves a target string by suffix -- `key == t or
       key.endswith("."+t)`. So a *leaf* name shared by both towers (`q_proj`)
       pulls the vision tower in even when we meant language-only, which would
       make `lora_vision=False` a lie and the ablation meaningless. Returning
       fully-qualified paths makes the split exact.
    """
    include_vision = CFG["lora_vision"] if include_vision is None else include_vision
    leaves = set(CFG["lora_targets"])
    lang, vis = [], []
    for name, mod in model.named_modules():
        if not isinstance(mod, torch.nn.Linear):
            continue
        parts = name.split(".")
        if any(p in PEFT_INTERNALS for p in parts):
            continue
        is_vision = any(p in parts for p in VISION_PREFIXES)
        if is_vision:
            vis.append(name)                       # every Linear in the tower
        elif name.rsplit(".", 1)[-1] in leaves:
            lang.append(name)                      # only the named projections
    if not vis:
        print(f"  !! no vision modules matched {VISION_PREFIXES}; inspect "
              "[n for n,_ in model.named_modules()] before trusting lora_vision")
    print(f"  language targets: {len(lang)} | vision targets: {len(vis)} "
          f"({'INCLUDED' if include_vision else 'frozen'})")
    if vis:
        print(f"  vision example:   {vis[0]}")
    return lang + (vis if include_vision else [])

def is_peft(m):
    """Whether LoRA is already attached, without importing a specific class."""
    return hasattr(m, "peft_config") or type(m).__name__.startswith("Peft")

def attach_lora(model, include_vision=None):
    # Re-running this cell after a later cell failed is the normal way to hit
    # this: the model from the first attempt is still wrapped, and wrapping it
    # again targets LoRA's own internals. Fail with the fix rather than letting
    # PEFT raise NoMatchingPeftModuleError about `qkv.base_layer`.
    if is_peft(model):
        raise RuntimeError(
            "LoRA is already attached to this model. Re-run the model-load cell "
            "in section 5 (`model, processor = load_base()`) to get a clean base "
            "model, then run this cell again.")
    targets = resolve_lora_targets(model, include_vision)
    cfg = LoraConfig(r=CFG["lora_r"], lora_alpha=CFG["lora_alpha"],
                     lora_dropout=CFG["lora_dropout"], bias="none",
                     target_modules=targets, task_type="CAUSAL_LM")
    m = get_peft_model(model, cfg); m.print_trainable_parameters(); return m

def build_messages(image, target=None):
    msgs = [{"role":"user","content":[{"type":"image","image":image},
                                      {"type":"text","text":INSTRUCTION}]}]
    if target is not None:
        msgs.append({"role":"assistant","content":[{"type":"text","text":target}]})
    return msgs

def render_chat(processor, messages, add_generation_prompt=False):
    """Apply the chat template with thinking mode OFF, always.

    Qwen3.5 enables thinking by default. Left on for this task it does not
    reason usefully -- it re-litigates ambiguous glyphs until it runs out of
    budget. The hosted runs showed 40,014 output tokens per rollout and one
    completion of 148,648 characters re-asserting a single word; here it put
    the baseline at CER 1.129 and length ratio 2.650 against an expected 0.168.
    Every measurement that produced 0.168 passed enable_thinking=False.

    Routed through one function because the collator renders the prompt twice --
    once with the target to build the training text, once without to find the
    mask boundary. If those two renderings disagree about thinking scaffolding,
    the prompt-length mask is wrong and SFT trains on a shifted target. One
    function, one behaviour.
    """
    kw = dict(tokenize=False, add_generation_prompt=add_generation_prompt)
    try:
        return processor.apply_chat_template(messages, enable_thinking=False, **kw)
    except TypeError:
        # Older processors do not forward unknown kwargs to the template.
        return processor.apply_chat_template(
            messages, chat_template_kwargs={"enable_thinking": False}, **kw)

from contextlib import contextmanager

@contextmanager
def padding_side(processor, side):
    """Temporarily set the tokenizer's padding side.

    Decoder-only generation requires LEFT padding. With right padding a short
    prompt becomes [tokens..., pad, pad, pad] and generation continues from the
    position *after* the pads rather than after the prompt's real last token --
    so the shortest items in a batch produce the worst output, and the damage
    scales with how uneven the batch is. Training wants the opposite: RIGHT
    padding, so labels line up with inputs.

    The Prime measurement of 0.168 sent one image per request and therefore
    never padded anything, which is why this never showed up there.
    """
    tok = processor.tokenizer
    prev = tok.padding_side
    tok.padding_side = side
    try:
        yield
    finally:
        tok.padding_side = prev

model, processor = load_base()
print("tokenizer default padding_side:", processor.tokenizer.padding_side,
      "(generation forces left, training forces right)")
print(type(model).__name__, "|", type(processor).__name__)

### 5a · Confirm thinking mode is off

Qwen3.5 reasons by default, and on this task it does not reason usefully — it
re-litigates ambiguous glyphs until the token budget runs out. Left on, the
hosted runs emitted 40,014 tokens per rollout; here it put the baseline at CER
1.129 with length ratio 2.650 against an expected 0.168.

This checks the rendered template directly rather than inferring it from a bad
CER an hour later.

In [ ]:
_probe = render_chat(processor, build_messages(SFT_EXAMPLES[0]["image"]),
                     add_generation_prompt=True)
_tail = _probe[-220:].replace("\n", "\\n")
print("rendered prompt tail:", _tail)

# An *empty* <think></think> pair is how Qwen3 signals thinking-off in some
# template versions, so its presence is fine. An unclosed <think> is not.
_open = _probe.count("<think>") - _probe.count("</think>")
assert _open == 0, (
    f"template left {_open} unclosed <think> -- thinking is ON. The model will "
    "reason instead of transcribing. Check that render_chat's enable_thinking "
    "kwarg reached this processor's template.")
print("\nthinking: OFF (no dangling <think>)  -- generation is probed in 6a")

## 6 · Generation and evaluation

`evaluate()` writes **one row per document** — not just aggregates — so any
comparison between arms can be recomputed or re-tested later without re-running
generation. Paired per-document records are what make the B-vs-C comparison
statistically usable.

In [ ]:
@torch.no_grad()
def generate_batch(model, processor, images, max_new=None, temperature=0.0, num_return=1):
    max_new = max_new or CFG["gen_max_new"]
    texts = [render_chat(processor, build_messages(im), add_generation_prompt=True)
             for im in images]
    # LEFT padding -- see padding_side(). Right padding silently corrupts every
    # item in the batch that is shorter than the longest one.
    with padding_side(processor, "left"):
        inputs = processor(text=texts, images=[[im] for im in images],
                           return_tensors="pt", padding=True).to(model.device)
    kw = dict(max_new_tokens=max_new, num_return_sequences=num_return,
              pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id)
    if temperature and temperature > 0:
        kw.update(do_sample=True, temperature=temperature, top_p=0.95)
    else:
        kw.update(do_sample=False)
    out = model.generate(**inputs, **kw)
    trimmed = out[:, inputs["input_ids"].shape[1]:]
    return [clean_output(t) for t in processor.batch_decode(trimmed, skip_special_tokens=True)]

def evaluate(model, processor, split="test", n=None, tag="eval", batch=4, save=True, step=None):
    ds = SPLITS[split]; n = n or len(ds)
    idx = list(range(min(n, len(ds)))); rows = []
    for i in range(0, len(idx), batch):
        chunk = [ds[j] for j in idx[i:i+batch]]
        imgs = [encode_image(c["image"]) for c in chunk]
        preds = generate_batch(model, processor, imgs, temperature=0.0)
        for c, p in zip(chunk, preds):
            r = cer_breakdown(c["ground_truth"], p)
            r.update(doc_id=c["doc_id"], pred=p, tag=tag, step=step)
            rows.append(r)
        if i % (batch*5) == 0:
            print(f"  {tag}: {i+len(chunk)}/{len(idx)}", end="\r")
    s = summarize(rows)
    print(f"\n[{tag}] n={s['n']} CER={s['cer']:.4f} sub={s['sub']:.4f} "
          f"del={s['dele']:.4f} ins={s['ins']:.4f} ratio={s['ratio']:.3f} empty={s['empty']:.3f}")
    if save:
        pd.DataFrame(rows).to_json(RUN/f"metrics/preds_{tag}.jsonl", orient="records", lines=True,
                                   force_ascii=False)
        jsonl_append({**s, "tag": tag, "step": step, "ts": time.time()}, RUN/"metrics/eval_log.jsonl")
    return s, rows

### 6a · One-document probe before the 100-document baseline

The full baseline is ~100 greedy generations. A misconfiguration costs that
whole run, so spend one generation first. **`ratio` is the diagnostic** — the
difference between "reads badly" and "is misconfigured" shows up there long
before it shows up in CER. Reading skill barely moves output length; thinking
mode, a missing stop condition, or a wrong prompt all push ratio past 2.

In [ ]:
_gen = generate_batch(model, processor, [SFT_EXAMPLES[0]["image"]], temperature=0.0)[0]
_b = cer_breakdown(SFT_EXAMPLES[0]["target"], _gen)
print(f"cer={_b['cer']:.3f} sub={_b['sub']:.3f} ins={_b['ins']:.3f} ratio={_b['ratio']:.3f}")
print(f"\nfirst 300 chars of output:\n{_gen[:300]}")
print(f"\nfirst 300 chars of ground truth:\n{SFT_EXAMPLES[0]['target'][:300]}")

if _b["ratio"] > 2.0:
    print("\n!! ratio > 2: the model is writing far more than the page contains.")
    print("   Thinking mode is the usual cause -- re-run 5a. Do NOT run 6b yet;")
    print("   it would spend 100 generations confirming a config bug.")
elif _b["ins"] > 0.5:
    print("\n!! insertions high at near-normal length: suspect the instruction")
    print("   text or clean_output, not the model's reading.")
else:
    print("\nlooks sane -- proceed to the batch-invariance check")

### 6a-ii · Batch invariance

The decisive test for a padding bug. One document decoded **alone** and the same
document decoded **inside a batch of four** must produce identical text —
greedy decoding is deterministic, and batching is supposed to be a performance
detail, not a semantic one.

If they differ, padding is corrupting the batch. The failure is asymmetric and
easy to miss: only items *shorter* than the longest in their batch are damaged,
so a one-document spot check looks perfect while the 100-document eval is wrong.
That is the shape of the discrepancy against Prime, where every request carried
a single image and nothing was ever padded.

In [ ]:
# Pick documents with very different ground-truth lengths: padding damage grows
# with the spread inside the batch, so a uniform batch can hide it.
_by_len = sorted(range(min(40, len(SFT_EXAMPLES))),
                 key=lambda i: len(SFT_EXAMPLES[i]["target"]))
_probe_i = _by_len[0]                      # shortest -> most padding, most damage
_others  = [_by_len[-1], _by_len[len(_by_len)//2], _by_len[-2]]

_alone = generate_batch(model, processor, [SFT_EXAMPLES[_probe_i]["image"]],
                        temperature=0.0)[0]
_batch = generate_batch(model, processor,
                        [SFT_EXAMPLES[i]["image"] for i in [_probe_i] + _others],
                        temperature=0.0)[0]

print(f"probe doc {SFT_EXAMPLES[_probe_i]['doc_id']} "
      f"(gt {len(SFT_EXAMPLES[_probe_i]['target'])} chars, shortest of {len(_by_len)})")
print(f"batch-mate gt lengths: {[len(SFT_EXAMPLES[i]['target']) for i in _others]}")
print(f"\nalone : {_alone[:120]!r}")
print(f"batched: {_batch[:120]!r}")

if _alone == _batch:
    print("\nIDENTICAL -- padding is correct, batched eval is trustworthy")
else:
    _ca = cer_breakdown(SFT_EXAMPLES[_probe_i]["target"], _alone)
    _cb = cer_breakdown(SFT_EXAMPLES[_probe_i]["target"], _batch)
    print(f"\n!! DIFFER -- batching changes the output.")
    print(f"   alone  : cer={_ca['cer']:.3f} ratio={_ca['ratio']:.3f}")
    print(f"   batched: cer={_cb['cer']:.3f} ratio={_cb['ratio']:.3f}")
    print("   Every batched number is suspect. Set evaluate(..., batch=1) to")
    print("   sidestep it, and check padding_side reached this processor.")

### 6b · Pre-training baseline on the test set

The reference every later number is measured against.

In [ ]:
base_summary, base_rows = evaluate(model, processor, "test", tag="arm0_base")
jdump(base_summary, RUN/"metrics/summary_arm0_base.json")

## 7 · Arm A — SFT on `sft_train`

Loss is masked to the assistant turn only: the model is scored on producing the
transcription, never on reproducing the instruction or the image placeholders.

Both the per-step training loss and periodic held-out CER are logged to disk.
Training loss alone is a poor guide here — it can fall steadily while CER does
nothing, which is exactly the case worth catching early.

In [ ]:
from torch.utils.data import Dataset as TorchDataset
from transformers import TrainerCallback
# Plain Trainer, not TRL's SFTTrainer. Every SFT-specific behaviour was already
# disabled here -- a custom collator plus skip_prepare_dataset -- so SFTTrainer
# contributed nothing but its dataset validation, which in TRL v1.x rejects
# torch Dataset subclasses outright ("must be a `Dataset` or `IterableDataset`").
# Our examples hold PIL images and are assembled by `collate`, so a torch
# Dataset is the right container.
from transformers import Trainer, TrainingArguments

class VLMOcrDataset(TorchDataset):
    def __init__(self, examples): self.ex = examples
    def __len__(self): return len(self.ex)
    def __getitem__(self, i): return self.ex[i]

def make_collator(processor):
    tok = processor.tokenizer
    def collate(batch):
        texts  = [render_chat(processor, build_messages(b["image"], b["target"]))
                  for b in batch]
        images = [[b["image"]] for b in batch]
        # RIGHT padding for training: labels are built by cloning input_ids, so
        # the two must align position-for-position. Pinned explicitly rather than
        # inherited, because generation flips this to "left".
        with padding_side(processor, "right"):
            enc = processor(text=texts, images=images, return_tensors="pt", padding=True,
                            truncation=True, max_length=CFG["sft_max_seq_len"])
        # Truncation that lands inside the image-token block is unrecoverable --
        # the processor raises a count mismatch rather than training on a damaged
        # sequence. Surface the real cause, since the processor's own message
        # blames truncation without saying what to set.
        if enc["input_ids"].shape[1] >= CFG["sft_max_seq_len"]:
            raise ValueError(
                f"sequence hit sft_max_seq_len ({CFG['sft_max_seq_len']}). Measured "
                f"max over this corpus is ~4,109 tokens at upscale 2 (image ~2,496 + "
                f"target up to ~1,563); you are at upscale {CFG['image_upscale']}. "
                "Raise CFG['sft_max_seq_len'] -- it is read per call, so no need to "
                "redefine the collator -- or lower CFG['image_upscale'].")
        labels = enc["input_ids"].clone()
        labels[labels == (tok.pad_token_id or tok.eos_token_id)] = -100
        # mask everything before the assistant turn so loss covers only the target
        for k, b in enumerate(batch):
            prompt = render_chat(processor, build_messages(b["image"]),
                                 add_generation_prompt=True)
            plen = len(tok(prompt, add_special_tokens=False)["input_ids"])
            labels[k, :plen] = -100
        for key in ("image_token_id", "video_token_id"):
            tid = getattr(processor, key, None) or getattr(getattr(model, "config", None), key, None)
            if tid is not None: labels[enc["input_ids"] == tid] = -100
        enc["labels"] = labels
        return enc
    return collate

class MetricLogger(TrainerCallback):
    """Streams every logged scalar to JSONL, and runs a held-out CER eval on a
    fixed cadence so the two curves can be read against each other."""
    def __init__(self, path, eval_every, eval_fn):
        self.path, self.eval_every, self.eval_fn = path, eval_every, eval_fn
        self.history = []
    def on_log(self, args, state, control, logs=None, **kw):
        if not logs: return
        rec = {"step": state.global_step, **{k: float(v) for k, v in logs.items()
               if isinstance(v, (int, float))}, "ts": time.time()}
        self.history.append(rec); jsonl_append(rec, self.path)
    def on_step_end(self, args, state, control, **kw):
        if self.eval_every and state.global_step > 0 and state.global_step % self.eval_every == 0:
            self.eval_fn(state.global_step)

In [ ]:
model = attach_lora(model)
model.config.use_cache = False
collate = make_collator(processor)

def periodic_eval(step):
    model.eval()
    try:
        evaluate(model, processor, "test", n=CFG["sft_eval_subset"],
                 tag=f"armA_step{step}", step=step)
    finally:
        model.train()

logger = MetricLogger(RUN/"metrics/sft_train_log.jsonl", CFG["sft_eval_every"], periodic_eval)

_est_total_steps = max(1, (len(SFT_EXAMPLES) * CFG["sft_epochs"])
                          // (CFG["sft_batch"] * CFG["sft_grad_accum"]))
print(f"estimated optimizer steps: {_est_total_steps}")

sft_args = TrainingArguments(
    output_dir=str(CKPT/"armA_sft"),
    num_train_epochs=CFG["sft_epochs"],
    per_device_train_batch_size=CFG["sft_batch"],
    gradient_accumulation_steps=CFG["sft_grad_accum"],
    learning_rate=CFG["sft_lr"],
    # warmup_ratio was removed from TrainingArguments in transformers 5.15.
    # An explicit step count avoids depending on whether this version reads a
    # float as a ratio; at ~180 total steps, 3% is 5.
    warmup_steps=max(1, int(CFG["sft_warmup_ratio"] * _est_total_steps)),
    lr_scheduler_type="cosine",
    logging_steps=1,
    save_strategy="steps", save_steps=50, save_total_limit=2,
    bf16=True, gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to=[],
    seed=CFG["seed"]
)

trainer = Trainer(model=model, args=sft_args,
                     train_dataset=VLMOcrDataset(SFT_EXAMPLES),
                     data_collator=collate, callbacks=[logger])

### 7a · Preflight on every example, then a smoke test

The smoke test runs two steps on four examples, which catches shape and masking
bugs. It does **not** catch anything that varies across the corpus — a four
document sample passed cleanly while a page five times larger was waiting in the
training set.

So measure the sequence length of every example first. It costs one processor
call per document and no GPU, and it either proves the budget holds or names the
document that breaks it.

In [ ]:
_tok = processor.tokenizer
_lens = []
for n, ex in enumerate(SFT_EXAMPLES):
    text = render_chat(processor, build_messages(ex["image"], ex["target"]))
    with padding_side(processor, "right"):
        ids = processor(text=[text], images=[[ex["image"]]], return_tensors="pt")["input_ids"]
    _lens.append((ids.shape[1], ex["doc_id"]))
    if n % 50 == 0: print(f"  {n}/{len(SFT_EXAMPLES)}", end="\r")

_lens.sort()
_tot = [L for L, _ in _lens]
print(f"sequence length over {len(_tot)} training examples:")
print(f"  median {_tot[len(_tot)//2]}  p90 {_tot[int(.9*len(_tot))]}  "
      f"max {_tot[-1]} (doc {_lens[-1][1]})")
_cap = CFG["sft_max_seq_len"]
_over = [(L, d) for L, d in _lens if L > _cap]
if _over:
    raise ValueError(
        f"{len(_over)} example(s) exceed sft_max_seq_len={_cap}, longest "
        f"{_over[-1][0]} tokens (doc {_over[-1][1]}). Truncation inside the "
        "image-token block raises rather than degrades, so this would fail "
        "mid-run. Lower CFG['max_image_tokens'] (currently "
        f"{CFG['max_image_tokens']}) or raise sft_max_seq_len.")
print(f"  all within sft_max_seq_len={_cap}  (headroom {_cap - _tot[-1]} tokens)")

Now the two-step smoke test.

In [ ]:
_smoke = Trainer(
    model=model,
    args=TrainingArguments(output_dir=str(CKPT/"_smoke"), max_steps=2,
                   per_device_train_batch_size=1, gradient_accumulation_steps=1,
                   logging_steps=1, bf16=True, gradient_checkpointing=True,
                   remove_unused_columns=False, report_to=[]),
    train_dataset=VLMOcrDataset(SFT_EXAMPLES[:4]), data_collator=collate)
_smoke.train()
print("\nsmoke OK -- loss is finite and backward() ran. Proceed.")

In [ ]:
t0 = time.time()
train_result = trainer.train()
print(f"\nSFT done in {(time.time()-t0)/60:.1f} min")

model.save_pretrained(RUN/"adapters/armA"); processor.save_pretrained(RUN/"adapters/armA")
jdump({k: float(v) for k, v in train_result.metrics.items()}, RUN/"metrics/armA_train_metrics.json")
pd.DataFrame(logger.history).to_csv(RUN/"metrics/armA_train_log.csv", index=False)

## 8 · SFT curves

Two panels, one axis each — never a dual-axis chart. Loss and CER live on
different scales and are plotted separately so neither is implied to track the
other.

In [ ]:
# Validated categorical palette (checked for CVD separation and contrast).
C = dict(loss="#2a78d6", cer="#eb6834", sub="#1baf7a", ins="#eda100", grid="#d8d8d4",
         ink="#0b0b0b", ink2="#52514e")
plt.rcParams.update({"figure.dpi": 130, "font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.edgecolor": C["ink2"],
                     "axes.labelcolor": C["ink2"], "text.color": C["ink"],
                     "xtick.color": C["ink2"], "ytick.color": C["ink2"]})

hist = pd.DataFrame(logger.history)
ev = pd.read_json(RUN/"metrics/eval_log.jsonl", lines=True)
ev_a = ev[ev.tag.str.startswith("armA_step")].sort_values("step") if len(ev) else pd.DataFrame()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

ax = axes[0]
if "loss" in hist:
    d = hist.dropna(subset=["loss"])
    ax.plot(d["step"], d["loss"], color=C["loss"], lw=1.2, alpha=.35)
    ax.plot(d["step"], d["loss"].rolling(15, min_periods=1).mean(), color=C["loss"], lw=2,
            label="training loss (15-step mean)")
    ax.legend(frameon=False, loc="upper right")
ax.set_xlabel("step"); ax.set_ylabel("cross-entropy"); ax.set_title("SFT training loss", loc="left")
ax.grid(axis="y", color=C["grid"], lw=.6); ax.set_axisbelow(True)

ax = axes[1]
if len(ev_a):
    for col, lab, col_c in [("cer","CER",C["cer"]),("sub","substitutions",C["sub"]),("ins","insertions",C["ins"])]:
        ax.plot(ev_a.step, ev_a[col], color=col_c, lw=2, marker="o", ms=4, label=lab)
        ax.annotate(f"{ev_a[col].iloc[-1]:.3f}", (ev_a.step.iloc[-1], ev_a[col].iloc[-1]),
                    textcoords="offset points", xytext=(6,0), color=col_c, fontsize=8, va="center")
    ax.axhline(base_summary["cer"], color=C["ink2"], lw=1, ls="--")
    ax.annotate(f"pre-SFT CER {base_summary['cer']:.3f}", (ev_a.step.iloc[0], base_summary["cer"]),
                textcoords="offset points", xytext=(0,5), color=C["ink2"], fontsize=8)
    ax.legend(frameon=False, loc="upper right")
ax.set_xlabel("step"); ax.set_ylabel("rate"); ax.set_ylim(bottom=0)
ax.set_title("Held-out error, decomposed", loc="left")
ax.grid(axis="y", color=C["grid"], lw=.6); ax.set_axisbelow(True)

plt.tight_layout(); plt.savefig(RUN/"figures/sft_curves.png", bbox_inches="tight", facecolor="white")
plt.show()

# table view -- the numbers behind the picture, also on disk
if len(ev_a):
    print(ev_a[["step","n","cer","sub","dele","ins","ratio","empty"]].round(4).to_string(index=False))
    ev_a.to_csv(RUN/"metrics/armA_eval_curve.csv", index=False)

### 8b · Arm A on the full test set

In [ ]:
model.eval()
armA_summary, armA_rows = evaluate(model, processor, "test", tag="armA_final")
jdump(armA_summary, RUN/"metrics/summary_armA.json")
print(f"\npre-SFT CER {base_summary['cer']:.4f}  ->  arm A CER {armA_summary['cer']:.4f}   "
      f"({(base_summary['cer']-armA_summary['cer'])/max(base_summary['cer'],1e-9):+.1%})")

### 8c · Resume after a disconnect

A Colab disconnect destroys the runtime and everything in `/content` — installs,
Python state, and the Trainer checkpoints under `ckpt_dir`. Drive survives, and
arm A's adapter plus every metric it produced live there, so **arm A never needs
retraining.**

To pick up here in a fresh session, run §1, §1a, §2, §3, §4, §5 and §6 — the
install, token, Drive mount and function definitions have to exist in the
runtime — then run the cell below **instead of** §7 and §8. Skip the
`load_base()` call at the end of §5; this replaces it.

Only arm A's weights are recoverable this way. If a *GRPO* run dies partway, its
optimizer state was in `ckpt_dir` and is gone; restart arm B from this same
adapter.

In [ ]:
ADAPTER = RUN/"adapters/armA"
assert ADAPTER.exists(), (
    f"no adapter at {ADAPTER}. If section 7 was interrupted before its save, "
    f"look for Trainer checkpoints under {CKPT}/armA_sft/checkpoint-* -- those "
    "are on local disk and only survive while the runtime lives.")

model, processor = load_base(adapter_path=ADAPTER, for_training=True)
processor.tokenizer.padding_side = "left"   # GRPO's scoring pass needs this
print("padding_side:", processor.tokenizer.padding_side)

# Rehydrate what sections 9-12 compare against, so the paired statistics still
# work without re-evaluating anything.
def _load_json(name):
    p = RUN/f"metrics/{name}"
    return json.loads(p.read_text()) if p.exists() else None

def _load_rows(tag):
    p = RUN/f"metrics/preds_{tag}.jsonl"
    if not p.exists(): return []
    return [json.loads(l) for l in p.read_text(encoding="utf-8").splitlines() if l.strip()]

base_summary = _load_json("summary_arm0_base.json")
armA_summary = _load_json("summary_armA.json")
base_rows, armA_rows = _load_rows("arm0_base"), _load_rows("armA_final")

for label, s in (("pre-SFT", base_summary), ("arm A", armA_summary)):
    print(f"  {label:<8} " + ("missing" if not s else
          f"n={s['n']} CER={s['cer']:.4f} sub={s['sub']:.4f} ins={s['ins']:.4f}"))
print(f"  rows reloaded: baseline {len(base_rows)}, arm A {len(armA_rows)}")
if base_summary and armA_summary:
    d = base_summary["cer"] - armA_summary["cer"]
    print(f"\narm A improvement: {d:+.4f} CER ({d/max(base_summary['cer'],1e-9):+.1%})")
if not armA_rows:
    print("\n!! no arm A per-document rows -- section 12's paired test needs them.")
    print("   Re-run 8b (full test eval) before section 12; it is ~15 min and")
    print("   does not require retraining.")

## 9 · GATE — `pass@16` on `rl_train`

**Decide here whether to run arm B at all.**

GRPO sharpens the distribution toward modes the policy already samples. The
useful quantity is the gap between the mean rollout and the best of k: that gap
*is* the headroom RL can convert into reliability.

Two gaps get reported. They are not the same quantity and the second is the one
that decides:

- `headroom` = mean-of-k − best-of-k. How much better the tail is than a typical
  sample.
- `greedy_minus_best` = greedy − best-of-k. Whether *sampling* beats *decoding* —
  i.e. whether there is anything RL could sharpen toward that greedy does not
  already produce.

`greedy_minus_best` is primary because it was measured to predict the failure.
On the 35B hosted run it was **0.028** before training, and the advantage standard
deviation observed at step 1 of that run was **0.032** — agreement to within 15%.
That advantage spread *is* the learning signal, so this probe is a cheap estimate
of it. Independently, [a controlled null on Qwen3-VL
4B/8B](https://arxiv.org/html/2607.12640) proposes the same screen and reports
GRPO gaining 22 points where the gap exists and nothing where it does not.

| `greedy_minus_best` | action |
|---|---|
| ≥ 0.10 | real tail → **run arm B** |
| 0.05 – 0.10 | marginal → only with a discretized reward (see §10) |
| < 0.05 | **skip arm B.** 0.028 and 0.015 both collapsed; believe it this time |

In [ ]:
@torch.no_grad()
def pass_at_k(model, processor, split="rl_train", k=None, n_docs=None, temperature=0.7):
    k = k or CFG["passk_k"]; n_docs = n_docs or CFG["passk_docs"]
    ds = SPLITS[split]; rows = []
    for i in range(min(n_docs, len(ds))):
        ex = ds[i]; im = encode_image(ex["image"])
        greedy = generate_batch(model, processor, [im], temperature=0.0)[0]
        samples = []
        for _ in range(k):
            samples += generate_batch(model, processor, [im], temperature=temperature)
        cers = [cer_breakdown(ex["ground_truth"], s)["cer"] for s in samples]
        g = cer_breakdown(ex["ground_truth"], greedy)["cer"]
        rows.append(dict(doc_id=ex["doc_id"], greedy=g, mean=float(np.mean(cers)),
                         best=float(np.min(cers)), worst=float(np.max(cers)),
                         std=float(np.std(cers)), n_better_than_greedy=int(sum(c < g for c in cers))))
        print(f"  {i+1}/{n_docs} greedy={g:.3f} mean={np.mean(cers):.3f} best={np.min(cers):.3f}", end="\r")
    d = pd.DataFrame(rows)
    res = dict(k=k, n_docs=len(d), greedy=d["greedy"].mean(), mean=d["mean"].mean(),
               best=d["best"].mean(), headroom=float(d["mean"].mean()-d["best"].mean()),
               greedy_minus_best=float(d["greedy"].mean()-d["best"].mean()),
               frac_docs_improved=float((d["best"] < d["greedy"]).mean()),
               frac_rollouts_better=float(d["n_better_than_greedy"].sum()/(len(d)*k)))
    d.to_csv(RUN/"metrics/passk_rl_train.csv", index=False); jdump(res, RUN/"metrics/passk_rl_train.json")
    return res, d

assert RL_AVAILABLE, ("rl_train was used for SFT, so pass@k on it measures "
                      "memorisation, not headroom. Skip sections 9-12.")
passk, passk_df = pass_at_k(model, processor)
print(f"""
greedy CER         {passk['greedy']:.4f}
mean-of-{passk['k']} CER     {passk['mean']:.4f}
best-of-{passk['k']} CER     {passk['best']:.4f}
headroom           {passk['headroom']:.4f}   (mean - best)
GREEDY - BEST      {passk['greedy_minus_best']:.4f}   <- the gate
docs improved      {passk['frac_docs_improved']:.1%}
rollouts better    {passk['frac_rollouts_better']:.1%}

reference: 35B hosted run had greedy-best 0.028 pre-training, advantage sd 0.032
at step 1, and collapsed to CER 1.500 on held-out data by step 25.
""")
_g = passk["greedy_minus_best"]
print("VERDICT:", "run arm B -- real tail to sharpen" if _g >= 0.10
      else "MARGINAL -- arm B only with a discretized reward (see section 10)" if _g >= 0.05
      else "SKIP arm B -- tail too thin; RL will drift as it did on hosted")

## 10 · Arm B — GRPO on `rl_train`

Reward is `1 − CER` against **human** ground truth, capped at 1.5 so a single
degenerate rollout cannot dominate its group's advantage, plus a weak smooth
length term. No cliff penalties: the hosted run's `−0.5` step function at ratio
2.0 was flat inside the band (nothing pulled the policy back as it drifted) and a
wall at the edge (which whipsawed it into emitting nothing).

Two deliberate departures from the hosted config, both aimed at the drift:
`grpo_lr` 5e-6 and `beta` 0.04 — a KL anchor ~40× stronger than the `kl_tau`
1e-3 those runs used.

> ⚠️ **This reward is still the continuous one, and `RL_review_v1.md` argues it is
> the wrong shape.** The case against it, briefly: a continuous per-page CER has a
> within-group spread (~0.03) at or below its own sampling noise floor, so the
> group comparison has nothing to bite on. It also makes the `zero_advantage`
> curriculum filter a no-op, because exact reward ties never occur across 8
> rollouts — the same silent pass-through that took GSM8K from 0.754 to 0.040 in
> [the phantom-advantage study](https://arxiv.org/html/2609.13866). Every
> successful OCR RL result in the literature uses a **discrete** reward instead:
> [olmOCR 2](https://arxiv.org/abs/2510.19817) scores pages by pass-fraction over
> binary unit tests, and argues edit distance "may penalize equally correct
> outputs."
>
> One caveat specific to TRL, and it matters: prime-rl mean-centres only
> (`advantages = rewards - rewards.mean()`), but TRL's `GRPOTrainer` defaults to
> `scale_rewards="group"`, which **divides by the group standard deviation**. That
> makes a low-variance reward strictly more dangerous here than it was on hosted —
> ε-scale differences become O(1) advantages, and turning the shaping coefficient
> down provably does not help, because z-scoring cancels it
> ([the Dark Room result](https://arxiv.org/html/2607.21273v1)).
>
> The trap: `loss_type="dr_grpo"` alone does *not* turn this off — the group
> scaling is a separate knob and stays on. `scale_rewards` is set to `"none"`
> below, with a fallback for older TRL versions that expect a bool.
>
> So: if the gate returns MARGINAL, discretize before running this. Per-line exact
> match over the ground-truth lines is the cheapest discrete reward available and
> needs no synthetic data. The continuous version is left here as the documented
> baseline, not as a recommendation.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

GT_BY_ID = {ex["doc_id"]: ex["ground_truth"] for ex in SPLITS["rl_train"]}
rl_reward_log = []

def ocr_reward(completions, doc_id=None, **kw):
    """1 - CER, plus -0.1*|ln(len ratio)|. Reward is always against human labels."""
    out = []
    for i, c in enumerate(completions):
        text = clean_output(c if isinstance(c, str) else c[0]["content"])
        gt = GT_BY_ID[doc_id[i]] if doc_id else ""
        b = cer_breakdown(gt, text)
        lr_pen = 0.0
        if b["ratio"] > 0:
            lr_pen = -0.1 * abs(math.log(max(b["ratio"], 1e-6)))
        r = (1.0 - b["cer"]) + lr_pen
        out.append(float(r))
        rl_reward_log.append(dict(doc_id=doc_id[i] if doc_id else None, reward=r, **b))
    return out

# Built with from_list, not .map(). Two separate problems, both schema-level:
#
#   1. `image` must be a LIST per sample. TRL's pixel-value slicing
#      (split_pixel_values_by_grid / _get_per_token_logps_and_entropies) assumes
#      a per-sample list and mis-slices a bare PIL image -> "shape mismatch:
#      value tensor of shape [3, N] cannot be broadcast to ... shape [3, M]".
#   2. .map() over SPLITS["rl_train"] inherits that split's `image` column type,
#      which is a scalar Image(). Returning a list for the same column name makes
#      datasets encode each PIL into {"bytes":..,"path":..} and then fail to fit a
#      dict into a scalar slot -> "only a single or a list of entries is supported
#      but got type=<class 'dict'>". remove_columns does not help: columns the
#      function returns are kept, schema and all.
#
# from_list starts from no prior schema, so features are inferred from what we
# actually hand it.
from datasets import Dataset as HFDataset

rl_ds = HFDataset.from_list([
    {"prompt": [{"role": "user", "content": [
         {"type": "image"}, {"type": "text", "text": INSTRUCTION}]}],
     "image": [encode_image(ex["image"])],
     "doc_id": ex["doc_id"]}
    for ex in SPLITS["rl_train"]
])
print(rl_ds.features)
_r0 = rl_ds[0]
print("image column ->", type(_r0["image"]).__name__,
      "of", type(_r0["image"][0]).__name__ if isinstance(_r0["image"], (list, tuple)) else "?")
# transformers' make_flat_list_of_images accepts a single image, a flat list, or
# a list of lists -- and nothing else. If the element above is a dict rather than
# an image, datasets handed back encoded bytes and the processor will raise
# "Could not make a flat list of images".

# Pin left padding for the rest of the run. padding_side() restores on exit, so
# without this the trainer's scoring pass re-processes prompts with the
# processor's default (right) while training uses left -- for Qwen-VL's M-RoPE
# that marks the image span at the wrong indices. See TRL issue ingambe/trl#126.
processor.tokenizer.padding_side = "left"
print("processor.tokenizer.padding_side =", processor.tokenizer.padding_side)

# With beta > 0, GRPOTrainer builds its KL reference by adding a disabled
# adapter named "ref" to the PEFT model. Re-running this cell after a failure
# finds it already there and PEFT raises "Adapter with name 'ref' already
# exists", so drop it first and keep the cell idempotent.
if "ref" in getattr(model, "peft_config", {}):
    model.delete_adapter("ref")
    print("removed stale 'ref' adapter from a previous attempt")

_grpo_kwargs = dict(
    output_dir=str(CKPT/"armB_grpo"),
    max_steps=CFG["grpo_steps"],
    per_device_train_batch_size=CFG["grpo_batch"],
    gradient_accumulation_steps=CFG["grpo_grad_accum"],
    num_generations=CFG["grpo_num_generations"],
    learning_rate=CFG["grpo_lr"],
    beta=CFG["grpo_beta"],
    temperature=CFG["grpo_temperature"],
    max_completion_length=CFG["grpo_max_new"],
    logging_steps=1, save_steps=20, save_total_limit=2,
    bf16=True, gradient_checkpointing=True, report_to=[], seed=CFG["seed"]
)

# Disable group-std advantage scaling. TRL >= 0.22 spells this scale_rewards="none";
# older versions take a bool; oldest have no knob at all. Probe rather than assume,
# because guessing wrong here silently leaves the amplification on -- and with a
# within-group spread of ~0.03 that is the difference between a tiny gradient and
# an O(1) one built entirely from noise.
import inspect
_fields = set(inspect.signature(GRPOConfig.__init__).parameters)
if "scale_rewards" in _fields:
    for _v in ("none", False):
        try:
            grpo_args = GRPOConfig(scale_rewards=_v, **_grpo_kwargs)
            print(f"  scale_rewards={_v!r} -> group-std scaling OFF (mean-centred advantages)")
            break
        except Exception as e:
            print(f"  scale_rewards={_v!r} rejected: {type(e).__name__}")
    else:
        grpo_args = GRPOConfig(**_grpo_kwargs)
        print("  !! could not disable scale_rewards; advantages ARE std-normalised")
else:
    grpo_args = GRPOConfig(**_grpo_kwargs)
    print("  !! this TRL has no scale_rewards knob; advantages ARE std-normalised.")
    print("     Upgrade TRL or expect the low-variance amplification described above.")

grpo_logger = MetricLogger(RUN/"metrics/grpo_train_log.jsonl", 0, lambda s: None)
grpo_trainer = GRPOTrainer(model=model, args=grpo_args, train_dataset=rl_ds,
                           reward_funcs=[ocr_reward], processing_class=processor,
                           callbacks=[grpo_logger])
print("GRPO configured. If this OOMs, drop grpo_batch to 4 or num_generations to 4.")

In [ ]:
model.train()
t0 = time.time(); grpo_trainer.train()
print(f"GRPO done in {(time.time()-t0)/60:.1f} min")

model.save_pretrained(RUN/"adapters/armB")
pd.DataFrame(rl_reward_log).to_csv(RUN/"metrics/armB_rollout_rewards.csv", index=False)
pd.DataFrame(grpo_logger.history).to_csv(RUN/"metrics/armB_train_log.csv", index=False)

armB_summary, armB_rows = evaluate(model, processor, "test", tag="armB_final")
jdump(armB_summary, RUN/"metrics/summary_armB.json")

### 10b · GRPO dynamics

The panel that matters is the second one. If `insertions` climbs while
`substitutions` is flat or rising, the policy is writing more rather than reading
better — the exact signature of both hosted failures. Stop the run if you see it.

In [ ]:
gl = pd.DataFrame(grpo_logger.history)
rl = pd.DataFrame(rl_reward_log)
if len(rl):
    rl["bucket"] = (np.arange(len(rl)) // max(1, len(rl)//40))
    agg = rl.groupby("bucket")[["reward","cer","sub","dele","ins","ratio"]].mean().reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    ax = axes[0]
    if "reward" in gl:
        d = gl.dropna(subset=["reward"])
        ax.plot(d["step"], d["reward"], color=C["loss"], lw=1.2, alpha=.35)
        ax.plot(d["step"], d["reward"].rolling(5, min_periods=1).mean(), color=C["loss"], lw=2,
                label="mean reward (5-step)")
        ax.legend(frameon=False)
    ax.set_xlabel("step"); ax.set_ylabel("reward"); ax.set_title("GRPO reward", loc="left")
    ax.grid(axis="y", color=C["grid"], lw=.6); ax.set_axisbelow(True)

    ax = axes[1]
    for col, lab, cc in [("sub","substitutions",C["sub"]),("ins","insertions",C["ins"]),("cer","CER",C["cer"])]:
        ax.plot(agg.bucket, agg[col], color=cc, lw=2, label=lab)
        ax.annotate(f"{agg[col].iloc[-1]:.3f}", (agg.bucket.iloc[-1], agg[col].iloc[-1]),
                    textcoords="offset points", xytext=(6,0), color=cc, fontsize=8, va="center")
    ax.legend(frameon=False); ax.set_xlabel("rollouts (binned)"); ax.set_ylabel("rate")
    ax.set_title("Error decomposition during RL", loc="left")
    ax.grid(axis="y", color=C["grid"], lw=.6); ax.set_axisbelow(True)
    plt.tight_layout(); plt.savefig(RUN/"figures/grpo_curves.png", bbox_inches="tight", facecolor="white")
    plt.show()
    agg.to_csv(RUN/"metrics/armB_binned.csv", index=False)
    print(agg.round(4).to_string(index=False))

## 11 · Arm C — the control

Reload the **arm A** adapter and continue SFT on `rl_train`. Same 250 documents
arm B saw, **unaugmented**, same starting point — supervised instead of
reinforced.

Without this, "arm B beat arm A" only shows that 250 extra documents helped.
The assert below enforces the parity that makes the comparison mean anything.

In [ ]:
del model, trainer, grpo_trainer; gc.collect(); torch.cuda.empty_cache()

modelC, processorC = load_base(adapter_path=str(RUN/"adapters/armA"), for_training=True)
modelC.config.use_cache = False
collateC = make_collator(processorC)

# NO augmentation here, regardless of CFG["USE_AUGMENTATION"]. Arm B (GRPO)
# rolls out on the 250 raw rl_train documents; if arm C trained on augmented
# copies of the same 250 it would see a larger effective dataset, and B-vs-C
# would stop being a comparison of *method* at a fixed data budget -- which is
# the only reason arm C exists. Augmentation belongs to arm A alone.
C_EXAMPLES = build_examples("rl_train", augment=False)

loggerC = MetricLogger(RUN/"metrics/armC_train_log.jsonl", 0, lambda s: None)
assert len(C_EXAMPLES) == len(SPLITS["rl_train"]), (
    "arm C must see exactly the documents arm B saw, once each")
argsC = TrainingArguments(output_dir=str(CKPT/"armC_sft"), num_train_epochs=CFG["c_epochs"],
                  per_device_train_batch_size=CFG["sft_batch"],
                  gradient_accumulation_steps=CFG["sft_grad_accum"],
                  learning_rate=CFG["c_lr"], lr_scheduler_type="cosine",
                  logging_steps=1, bf16=True, gradient_checkpointing=True,
                  remove_unused_columns=False,
                  report_to=[], seed=CFG["seed"])
trainerC = Trainer(model=modelC, args=argsC, train_dataset=VLMOcrDataset(C_EXAMPLES),
                      data_collator=collateC, callbacks=[loggerC])
trainerC.train()
modelC.save_pretrained(RUN/"adapters/armC")
pd.DataFrame(loggerC.history).to_csv(RUN/"metrics/armC_train_log.csv", index=False)

modelC.eval()
armC_summary, armC_rows = evaluate(modelC, processorC, "test", tag="armC_final")
jdump(armC_summary, RUN/"metrics/summary_armC.json")

## 12 · Paired comparison

Every arm scored the same 100 documents, so differences are tested **per
document**. A paired test removes between-document variance, which here (CER std
≈ 0.18) dwarfs the effects being measured — an unpaired comparison on n=100 would
not resolve them.

In [ ]:
from scipy import stats

def paired(rows_a, rows_b, name_a, name_b):
    a = pd.DataFrame(rows_a).set_index("doc_id")["cer"]
    b = pd.DataFrame(rows_b).set_index("doc_id")["cer"]
    common = a.index.intersection(b.index); a, b = a[common], b[common]
    d = a - b
    t, p = stats.ttest_rel(a, b)
    w = stats.wilcoxon(a, b) if len(common) > 10 else None
    return dict(comparison=f"{name_a} -> {name_b}", n=len(common),
                cer_a=float(a.mean()), cer_b=float(b.mean()), delta=float(d.mean()),
                pct_change=float(d.mean()/max(a.mean(),1e-9)),
                docs_improved=float((b < a).mean()),
                cohens_d=float(d.mean()/max(d.std(),1e-9)),
                t_p=float(p), wilcoxon_p=float(w.pvalue) if w else None)

comparisons = [
    paired(base_rows, armA_rows, "base", "armA_SFT"),
    paired(armA_rows, armB_rows, "armA_SFT", "armB_SFT+RL"),
    paired(armA_rows, armC_rows, "armA_SFT", "armC_SFT+SFT"),
    paired(armC_rows, armB_rows, "armC_SFT+SFT", "armB_SFT+RL"),   # <- the experiment
]
comp = pd.DataFrame(comparisons)
comp.to_csv(RUN/"metrics/paired_comparisons.csv", index=False)
jdump(comparisons, RUN/"metrics/paired_comparisons.json")
print(comp.round(4).to_string(index=False))

summ = pd.DataFrame([{"arm":"base (no training)", **base_summary},
                     {"arm":"A: SFT", **armA_summary},
                     {"arm":"B: SFT+RL", **armB_summary},
                     {"arm":"C: SFT+SFT", **armC_summary}])
summ.to_csv(RUN/"metrics/arm_summaries.csv", index=False)
print("\n", summ.round(4).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

ax = axes[0]
arms = summ["arm"].tolist(); vals = summ["cer"].tolist()
bars = ax.bar(range(len(arms)), vals, color=[C["ink2"], C["loss"], C["cer"], C["sub"]],
              width=.62, zorder=3)
for i, v in enumerate(vals):
    ax.annotate(f"{v:.3f}", (i, v), textcoords="offset points", xytext=(0,4),
                ha="center", fontsize=9, color=C["ink"])
ax.set_xticks(range(len(arms))); ax.set_xticklabels([a.split(":")[0] for a in arms])
ax.set_ylabel("CER (lower is better)"); ax.set_title("Held-out CER by arm", loc="left")
ax.grid(axis="y", color=C["grid"], lw=.6); ax.set_axisbelow(True)

ax = axes[1]
stack = summ[["sub","dele","ins"]].values
bottom = np.zeros(len(summ))
for j,(lab,cc) in enumerate([("substitutions",C["sub"]),("deletions",C["loss"]),("insertions",C["ins"])]):
    ax.bar(range(len(summ)), stack[:,j], bottom=bottom, color=cc, width=.62,
           label=lab, zorder=3, edgecolor="white", linewidth=2)
    bottom += stack[:,j]
ax.set_xticks(range(len(summ))); ax.set_xticklabels([a.split(":")[0] for a in arms])
ax.set_ylabel("rate"); ax.legend(frameon=False, fontsize=8)
ax.set_title("Where the errors are", loc="left")
ax.grid(axis="y", color=C["grid"], lw=.6); ax.set_axisbelow(True)

plt.tight_layout(); plt.savefig(RUN/"figures/arm_comparison.png", bbox_inches="tight", facecolor="white")
plt.show()

## 13 · Readout

**The headline is the last row of the comparison table: `armC → armB`.** That is
RL versus the same data spent on supervision.

| result | interpretation |
|---|---|
| B better than C, p < 0.05 | RL adds real value beyond the data it consumed |
| B ≈ C | SFT→RL is not worth the complexity here; ship arm C |
| B worse than C | RL is degrading a good policy — consistent with both hosted runs |

Read the second panel too. If arm B's improvement comes from **insertions**
falling rather than **substitutions**, it learned to stop rambling, not to read
better — a real but much less interesting result than it looks.

**Reference points for arm A's CER**, on this same benchmark: 0.168 is the
untrained high-res baseline, **0.159 is TrOCR** (the bar), 0.032 is
Gemini-2.5-Pro. A frontier VLM reads these images at 0.022, so the information is
in the pixels and any shortfall is model capability, not data.

**If arm A barely moved,** in order of what to check:

1. **Confirm the vision tower actually trained.** `lora_vision=True` is the
   default now, but `resolve_lora_targets` prints how many vision modules it
   matched — if that count is 0, the prefix list missed this architecture and you
   silently trained language-only. That single line is the check.
2. **Run the language-only ablation** (`lora_vision=False`). If it matches, the
   encoder was never the bottleneck and the ceiling is elsewhere.
3. **Suspect data volume before architecture.** 479 page-level examples is below
   the range where this is known to work: comparable low-resource results used 911
   samples (Jawi, Qwen2-VL-2B → CER 0.0866) and 60k (Manchu). The corpus holds
   **9,982 ground-truth-aligned lines**, and projection-profile segmentation
   recovers an exact line count on ~61% of documents — roughly 5,000 usable
   line-level examples, at ~40 output tokens each instead of ~1,000. That is the
   next lever, and it is larger than any hyperparameter here.
4. **Resolution, again.** 2× on high-res is the measured operating point but the
   sweep was coarse. 1× and 3× are one config change each.

Everything written to disk under `run_dir`:

```
metrics/
  sft_train_log.jsonl        per-step loss/lr/grad-norm (arm A)
  armA_train_log.csv         same, tabular
  armA_eval_curve.csv        periodic held-out CER during SFT
  eval_log.jsonl             every evaluation, all arms
  preds_<tag>.jsonl          per-document predictions + CER breakdown
  passk_rl_train.{csv,json}  the arm-B gate
  armB_rollout_rewards.csv   per-rollout reward + decomposition
  armB_binned.csv            binned GRPO dynamics
  arm_summaries.csv          one row per arm
  paired_comparisons.{csv,json}
figures/
  sft_curves.png  grpo_curves.png  arm_comparison.png
adapters/  armA/  armB/  armC/
```

---

# 14 · Arm B′ — GRPO with a discrete line reward

Self-contained retry of the RL arm. **Nothing above this needs re-running except
setup and §8c**; the cell list is in §14a.

## Why arm B failed, and what changed

Arm B reached CER 1.4077 with `ins` 5.2160 and a degenerate `usr` token loop. The
rollout log shows that was *not* an RL collapse — reward rose from −0.185 to
**+0.928**, and the final rollouts scored CER 0.059 at ratio 1.038. The policy
worked. It was evaluated on inputs it had never seen:

| | images | CER |
|---|---|---:|
| arm B final rollouts | squared 1024×1024 | 0.059 |
| arm B held-out eval | native aspect, `encode_image` | 1.4077 |

GRPO spent 60 steps re-adapting the policy to a squared-image transform
introduced to work around a TRL shape error, and the eval then fed it native
aspect ratios. Same weights, different distribution.

Four changes here:

1. **No image transform.** `encode_image` is untouched, so rollouts and eval see
   identical inputs and arm A's 0.0315 stays a valid comparison. The uniform
   image-token count TRL needs comes from `batch == num_generations`, which puts
   one document's rollout group in each forward pass.
2. **Discrete reward: F1 over exact line matches.** Measured on arm C's
   predictions, **46.9%** of ground-truth lines come back exactly right (per-doc
   median 0.477, p10 0.250, p90 0.750, only 6/100 documents degenerate). A
   Bernoulli variable at p≈0.47 is at maximum variance, the same regime as the
   one successful prime-rl reference run (56.25%). One line of twelve differing
   between rollouts moves F1 by 0.083; the same difference moves page CER by
   ~0.005, which was below its own noise floor.
3. **No reward plateau.** F1 is bounded in [0,1] with no cap, so over-generation
   is never free — extra lines cut precision directly. That was the mechanism
   behind RL-1b, RL-2 and arm B, and it removes the need for a length penalty
   competing with the primary term.
4. **Start from arm A, not arm C.** Arm C trained on `rl_train`; RL on the same
   documents would optimize against memorised data. Starting from arm A keeps
   B′ vs C a fair comparison at a fixed data budget.

## 14a · What to run

From a cold runtime, in order:

| cells | what |
|---|---|
| 2 | install (also removes torchao) |
| 4, 5 | HF token + dataset access |
| 6 | imports, `seed_all` |
| 8 | `CFG`, Drive mount, `RUN` / `CKPT` |
| 10 | metrics: `normalize`, `cer_breakdown`, `clean_output` |
| 12, 13 | `SPLITS`, `encode_image` (~3 min) |
| 15 | `load_base`, `render_chat`, `padding_side` — **let the trailing `load_base()` run or interrupt it; §8c reloads anyway** |
| 19 | `generate_batch`, `evaluate` |
| 27 | `MetricLogger`, `VLMOcrDataset` (definitions only, no training) |
| **39** | **§8c — loads arm A's adapter from Drive** |
| then | every cell in §14 below, in order |

Skip 17, 21, 23, 25 (probes and the baseline, already measured), 28–37 (arm A
training), and 41–51 (the old gate, arm B, arm C).

In [ ]:
from collections import Counter

def line_f1(pred, gt):
    """F1 over exact line matches, as a multiset so repeated lines count once each.

    Precision is what makes this safe: emitting extra lines to catch more
    matches lowers it, so over-generation costs reward instead of being free.
    That is the property every previous reward here lacked -- CER capped at 1.5
    went flat past ~2.5x length, and the policy ran away into the plateau.
    """
    g = Counter(x for x in (normalize(l) for l in (gt or "").splitlines()) if x)
    p = Counter(x for x in (normalize(l) for l in (pred or "").splitlines()) if x)
    hits = sum((g & p).values())
    n_g, n_p = sum(g.values()), sum(p.values())
    if n_g == 0 or n_p == 0 or hits == 0:
        return 0.0
    prec, rec = hits / n_p, hits / n_g
    return 2 * prec * rec / (prec + rec)

# Behaviour checks before it is wired to a reward.
_gt3 = "alpha\nbeta\ngamma"
for _name, _hyp, _want in [
    ("perfect",        "alpha\nbeta\ngamma", 1.0),
    ("one line wrong", "alpha\nbeta\nGAMMA", 2*(2/3)*(2/3)/((2/3)+(2/3))),
    ("one missing",    "alpha\nbeta",         2*1.0*(2/3)/(1.0+(2/3))),
    ("duplicated",     "alpha\nalpha\nbeta\ngamma", 2*(3/4)*1.0/((3/4)+1.0)),
    ("padded garbage", "alpha\nbeta\ngamma\nxx\nyy\nzz", 2*0.5*1.0/(0.5+1.0)),
    ("empty",          "",                     0.0),
]:
    _got = line_f1(_hyp, _gt3)
    print(f"  {_name:<16} F1={_got:.3f} expected {_want:.3f} "
          f"{'OK' if abs(_got-_want) < 1e-6 else 'MISMATCH'}")
print("\nnote the 'padded garbage' row: recall stays 1.0 but F1 drops to 0.667,")
print("which is the plateau fix -- extra output is penalised, not free.")

## 14b · Gate — within-group F1 spread

The 46.9% figure says the reward is well positioned *across* documents. It does
not say rollouts differ *on the same* document, and that within-group spread is
what GRPO actually consumes.

This is the same probe that predicted every previous failure, in F1 units.
Higher F1 is better, so the useful direction is now best-of-k **above** greedy.

| within-group sd | action |
|---|---|
| ≥ 0.08 | real signal → run it |
| 0.03 – 0.08 | marginal; still better than the 0.028 CER spread ever was |
| < 0.03 | the same dead end in different units → **stop here** |

In [ ]:
@torch.no_grad()
def f1_gate(model, processor, split="rl_train", k=8, n_docs=10, temperature=0.7):
    ds, rows = SPLITS[split], []
    for i in range(min(n_docs, len(ds))):
        ex = ds[i]; im = encode_image(ex["image"])
        greedy = generate_batch(model, processor, [im], temperature=0.0)[0]
        samples = []
        for _ in range(k):
            samples += generate_batch(model, processor, [im], temperature=temperature)
        f = [line_f1(s, ex["ground_truth"]) for s in samples]
        g = line_f1(greedy, ex["ground_truth"])
        rows.append(dict(doc_id=ex["doc_id"], greedy=g, mean=float(np.mean(f)),
                         best=float(np.max(f)), worst=float(np.min(f)),
                         sd=float(np.std(f)), n_better=int(sum(x > g for x in f))))
        print(f"  {i+1}/{n_docs} greedy={g:.3f} mean={np.mean(f):.3f} "
              f"best={np.max(f):.3f} sd={np.std(f):.3f}   ", end="\r")
    d = pd.DataFrame(rows)
    res = dict(k=k, n_docs=len(d), greedy=float(d["greedy"].mean()),
               mean=float(d["mean"].mean()), best=float(d["best"].mean()),
               within_group_sd=float(d["sd"].mean()),
               best_minus_greedy=float(d["best"].mean() - d["greedy"].mean()),
               frac_rollouts_better=float(d["n_better"].sum() / (len(d) * k)))
    d.to_csv(RUN/"metrics/f1_gate_rl_train.csv", index=False)
    jdump(res, RUN/"metrics/f1_gate_rl_train.json")
    return res, d

f1g, f1g_df = f1_gate(model, processor)
print(f"""
greedy F1          {f1g['greedy']:.4f}
mean-of-{f1g['k']} F1      {f1g['mean']:.4f}
best-of-{f1g['k']} F1      {f1g['best']:.4f}
WITHIN-GROUP SD    {f1g['within_group_sd']:.4f}   <- the gate
best - greedy      {f1g['best_minus_greedy']:+.4f}
rollouts > greedy  {f1g['frac_rollouts_better']:.1%}
""")
_sd = f1g["within_group_sd"]
print("VERDICT:", "RUN -- real within-group signal" if _sd >= 0.08
      else "MARGINAL -- proceed only if you want the measurement" if _sd >= 0.03
      else "STOP -- spread below the noise floor, same dead end in new units")

## 14c · GRPO setup

Every fix from the arm B post-mortem, in one place.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

GT_BY_ID2 = {ex["doc_id"]: ex["ground_truth"] for ex in SPLITS["rl_train"]}
rl2_log, _think_seen = [], 0

def line_f1_reward(completions, doc_id=None, **kw):
    """Primary reward. Raises rather than scoring against an empty reference.

    The original ocr_reward fell back to `gt = ""` when doc_id was absent, which
    silently makes every reward a constant -- zero advantage, no learning, and no
    error message. If TRL ever stops forwarding dataset columns, fail loudly.
    """
    global _think_seen
    if doc_id is None:
        raise RuntimeError(
            "TRL did not forward `doc_id`; the reward cannot find its ground "
            "truth. Check that rl_ds still has a doc_id column.")
    out = []
    for i, c in enumerate(completions):
        raw = c if isinstance(c, str) else c[0]["content"]
        if "<think>" in (raw or ""):
            _think_seen += 1
        text = clean_output(raw)
        gt = GT_BY_ID2[doc_id[i]]
        r = line_f1(text, gt)
        out.append(float(r))
        b = cer_breakdown(gt, text)       # weight 0, monitoring only
        rl2_log.append(dict(doc_id=doc_id[i], reward=r, **b))
    return out

# Bare PIL per sample, built with from_list.
#   - A LIST per sample makes TRL's own wrapping produce [[[PIL]]], three levels
#     deep, and make_flat_list_of_images accepts at most two.
#   - .map() over SPLITS inherits that split's scalar Image() feature, so a
#     returned list gets encoded to dicts and fails. from_list infers fresh.
from datasets import Dataset as HFDataset

rl_ds2 = HFDataset.from_list([
    {"prompt": [{"role": "user", "content": [
         {"type": "image"}, {"type": "text", "text": INSTRUCTION}]}],
     "image": encode_image(ex["image"]),
     "doc_id": ex["doc_id"]}
    for ex in SPLITS["rl_train"]
])
print(rl_ds2)
print("image ->", type(rl_ds2[0]["image"]).__name__, rl_ds2[0]["image"].size)

In [ ]:
# TRL renders `prompt` with the processor's own chat template and never passes
# enable_thinking=False. If that template reasons by default, rollouts reason
# while the eval does not -- a train/eval mismatch, and reasoning text stripped
# by clean_output would cost nothing in reward.
#
# Qwen disables thinking by emitting a PRE-CLOSED block after the generation
# prompt:
#     {%- if enable_thinking is defined and enable_thinking is false %}
#         {{- '<think>\n\n</think>\n\n' }}
# so "off" hands the model an already-closed <think></think> and it goes
# straight to content. Left on, Qwen3.5 opens a <think> and never closes it --
# the dangling tag the check below counts.
#
# Patching the default onto the processor is version-agnostic: it does not
# depend on the template's text, and it covers TRL's call sites as well as ours.
def _force_thinking_off(obj):
    orig = getattr(obj, "apply_chat_template", None)
    if orig is None or getattr(orig, "_thinking_patched", False):
        return False
    def patched(*a, **kw):
        kw.setdefault("enable_thinking", False)
        return orig(*a, **kw)
    patched._thinking_patched = True
    obj.apply_chat_template = patched
    return True

for _obj, _label in ((processor, "processor"), (processor.tokenizer, "tokenizer")):
    print(f"  {_label}: {'patched' if _force_thinking_off(_obj) else 'already patched'}")

_trl_prompt = processor.apply_chat_template(rl_ds2[0]["prompt"], tokenize=False,
                                             add_generation_prompt=True)
_dangling = _trl_prompt.count("<think>") - _trl_prompt.count("</think>")
print("TRL-rendered tail:", repr(_trl_prompt[-200:]))
print("dangling <think>:", _dangling)
assert _dangling == 0, (
    f"TRL's rollout prompt still leaves {_dangling} unclosed <think> after "
    "patching. The template may not accept the enable_thinking kwarg on this "
    "version -- print processor.chat_template and patch the text directly. Do "
    "NOT train through this: an unverified mismatch here is what made arm B "
    "uninterpretable.")

# Closed-and-empty is the intended shape; present is fine, dangling is not.
print(f"\nthinking: OFF in TRL's rollout path "
      f"(<think> blocks: {_trl_prompt.count('<think>')}, all closed)")

# And confirm TRL's render now matches the one the eval path uses, since the
# whole point is that rollouts and evaluation see the same prompt.
_our = render_chat(processor, build_messages(rl_ds2[0]["image"]),
                   add_generation_prompt=True)
print("matches render_chat (eval path):", _trl_prompt == _our)
if _trl_prompt != _our:
    print("  TRL :", repr(_trl_prompt[-120:]))
    print("  OURS:", repr(_our[-120:]))
    print("  -> a residual difference here means rollouts and eval still differ")

In [ ]:
processor.tokenizer.padding_side = "left"      # scoring pass must match training
if "ref" in getattr(model, "peft_config", {}):
    model.delete_adapter("ref")                # beta>0 re-adds it; cell stays idempotent
    print("removed stale 'ref' adapter")

_g2 = dict(
    output_dir=str(CKPT/"armB2_grpo"),
    max_steps=60,
    # batch == num_generations puts ONE document's rollout group in each forward
    # pass, so every image in a batch is the same image and the token count is
    # uniform. That is what the shape mismatch was about -- 16 images of two
    # different sizes in one slice -- and it is why no image transform is needed.
    per_device_train_batch_size=8,
    num_generations=8,
    gradient_accumulation_steps=8,             # raise this for effective batch
    learning_rate=5e-6,
    beta=0.04,                                 # KL anchor to the arm A policy
    temperature=0.7,
    max_completion_length=1536,                # GT max is 1,563 tokens; p90 is 726
    logging_steps=1, save_steps=20, save_total_limit=2,
    bf16=True, gradient_checkpointing=True, report_to=[], seed=CFG["seed"],
)

# TRL defaults to scale_rewards="group", dividing advantages by the group std.
# prime-rl mean-centres only. With a low-variance reward the division turns
# eps-scale noise into O(1) advantages, and lowering the shaping coefficient
# provably does not help because z-scoring cancels it. loss_type="dr_grpo"
# does NOT disable this; it is a separate knob.
import inspect
if "scale_rewards" in set(inspect.signature(GRPOConfig.__init__).parameters):
    for _v in ("none", False):
        try:
            grpo2_args = GRPOConfig(scale_rewards=_v, **_g2)
            print(f"scale_rewards={_v!r} -> group-std scaling OFF"); break
        except Exception as e:
            print(f"scale_rewards={_v!r} rejected: {type(e).__name__}")
    else:
        grpo2_args = GRPOConfig(**_g2); print("!! group-std scaling is ON")
else:
    grpo2_args = GRPOConfig(**_g2); print("!! no scale_rewards knob in this TRL")

grpo2_logger = MetricLogger(RUN/"metrics/armB2_train_log.jsonl", 0, lambda s: None)
grpo2_trainer = GRPOTrainer(model=model, args=grpo2_args, train_dataset=rl_ds2,
                            reward_funcs=[line_f1_reward], processing_class=processor,
                            callbacks=[grpo2_logger])
print("\nconfigured. If this OOMs, drop num_generations and "
      "per_device_train_batch_size together -- they must stay equal.")

## 14d · Train

Watch **reward**, not loss. GRPO's loss is a surrogate whose advantages are
mean-centred within each group, so it hovers near zero and fluctuates regardless
of policy quality — a rising loss means nothing here.

The first thing to check is that step-1 rollout reward is near the gate's
`mean-of-8` figure. Arm B's rollouts opened at `ins` 7.4 when arm A's greedy was
0.0094, and that 300× discrepancy was the signal that the rollout path differed
from the eval path. It was visible at step 1 and went unread.

In [ ]:
t0 = time.time()
grpo2_trainer.train()
print(f"\nGRPO done in {(time.time()-t0)/60:.1f} min")

model.save_pretrained(RUN/"adapters/armB2")
d2 = pd.DataFrame(rl2_log)
d2.to_csv(RUN/"metrics/armB2_rollout_rewards.csv", index=False)
pd.DataFrame(grpo2_logger.history).to_csv(RUN/"metrics/armB2_train_log.csv", index=False)

k = max(1, len(d2)//10)
print(f"rollouts: {len(d2)}   <think> seen in {_think_seen} completions")
print(f"first decile reward {d2['reward'][:k].mean():.3f} -> "
      f"last decile {d2['reward'][-k:].mean():.3f}")
print(f"first decile ratio  {d2['ratio'][:k].mean():.3f} -> "
      f"last decile {d2['ratio'][-k:].mean():.3f}")
print(f"first decile cer    {d2['cer'][:k].mean():.3f} -> "
      f"last decile {d2['cer'][-k:].mean():.3f}")

## 14e · Held-out evaluation and the comparison that matters

Same `evaluate()`, same `encode_image`, same test split as arms A and C — so
these numbers are directly comparable, which was exactly what arm B's were not.

**B′ vs C is the experiment.** Both consumed the same 250 `rl_train` documents
starting from arm A; one by reinforcement, one by supervision.

In [ ]:
model.eval()
armB2_summary, armB2_rows = evaluate(model, processor, "test", tag="armB2_final")
jdump(armB2_summary, RUN/"metrics/summary_armB2.json")

def _load(name):
    p = RUN/f"metrics/{name}"
    return json.loads(p.read_text()) if p.exists() else None

_rows = []
for label, s in (("base", _load("summary_arm0_base.json")),
                 ("A: SFT", _load("summary_armA.json")),
                 ("C: SFT+SFT", _load("summary_armC.json")),
                 ("B2: SFT+RL(F1)", armB2_summary)):
    if s: _rows.append(dict(arm=label, **{k: s[k] for k in
                       ("n","cer","sub","dele","ins","ratio","empty")}))
tbl = pd.DataFrame(_rows)
print(tbl.round(4).to_string(index=False))
tbl.to_csv(RUN/"metrics/arm_summaries_v2.csv", index=False)

# Paired per-document comparison; reuses `paired` from section 12 when present.
armA_rows = [json.loads(l) for l in
             (RUN/"metrics/preds_armA_final.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
armC_p = RUN/"metrics/preds_armC_final.jsonl"
armC_rows = ([json.loads(l) for l in armC_p.read_text(encoding="utf-8").splitlines() if l.strip()]
             if armC_p.exists() else [])

def _paired(a_rows, b_rows, label):
    from scipy import stats
    a = {r["doc_id"]: r["cer"] for r in a_rows}
    b = {r["doc_id"]: r["cer"] for r in b_rows}
    ids = sorted(set(a) & set(b))
    da = np.array([a[i] for i in ids]); db = np.array([b[i] for i in ids])
    diff = da - db
    t = stats.ttest_rel(da, db); w = stats.wilcoxon(da, db)
    d = diff.mean() / diff.std(ddof=1) if diff.std(ddof=1) > 0 else 0.0
    print(f"{label:<28} n={len(ids)} cer_a={da.mean():.4f} cer_b={db.mean():.4f} "
          f"delta={diff.mean():+.4f} improved={100*(diff>0).mean():.0f}% "
          f"d={d:+.3f} t_p={t.pvalue:.4g} w_p={w.pvalue:.4g}")

print()
_paired(armA_rows, armB2_rows, "armA -> armB2 (RL)")
if armC_rows:
    _paired(armC_rows, armB2_rows, "armC -> armB2 (RL vs SFT)")

## 14f · Reading the result

| outcome | interpretation |
|---|---|
| B′ below C, significant | RL adds something supervision on the same data does not. The discrete reward was the missing piece. |
| B′ ≈ C | RL matches supervision at equal data budget — no advantage, but no longer a failure. |
| B′ ≈ A | RL did nothing; the gate's prior was right. |
| B′ well above A | degradation. Check `ratio` and `ins` first: if they rose, the plateau is back despite F1. |

Check the decomposition either way. The remaining error in arm C is
`sub 0.0047 / del 0.0155 / ins 0.0087`, so **deletions are the largest
component** and the line reward targets them directly — a dropped line costs
recall. If B′ improves, deletions falling is the mechanism to confirm.

A note on what this run can and cannot settle. A null result here is informative
in a way arm B's was not: same inputs, same eval, a reward whose resolution was
measured in advance, and a gate reading recorded before training. A null would
say RL has nothing to add *at this performance level* — not that the earlier
setup was broken, which is all arm B established.